# 📧 Email Spam Classifier — Exploratory Notebook
This notebook walks through the full ML pipeline:
1. Data exploration & visualization
2. Text preprocessing
3. Feature engineering (TF-IDF)
4. Model training & comparison
5. Error analysis
6. Final model selection

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

from sklearn.metrics import ConfusionMatrixDisplay
from spam_classifier import (
    load_sample_data, preprocess_text,
    train_and_evaluate, build_pipelines, predict
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ Imports complete')

## 1. Load & Inspect Data

In [ ]:
# Load data — swap with load_data('your_file.csv') for real datasets
df = load_sample_data()
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
labels_map = {0: 'Ham', 1: 'Spam'}
axes[0].bar(['Ham', 'Spam'], [counts[0], counts[1]], color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=2)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie([counts[0], counts[1]], labels=['Ham', 'Spam'], colors=['#2ecc71', '#e74c3c'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Spam vs Ham Ratio', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis
df['text_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, name in [(0, '#2ecc71', 'Ham'), (1, '#e74c3c', 'Spam')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['text_length'], alpha=0.6, color=color, label=name, bins=20)
    axes[1].hist(subset['word_count'], alpha=0.6, color=color, label=name, bins=20)

axes[0].set_title('Text Length Distribution', fontweight='bold')
axes[0].set_xlabel('Characters')
axes[0].legend()

axes[1].set_title('Word Count Distribution', fontweight='bold')
axes[1].set_xlabel('Words')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, label, title, color in [
    (axes[0], 1, 'SPAM Word Cloud', 'Reds'),
    (axes[1], 0, 'HAM Word Cloud', 'Greens'),
]:
    text = ' '.join(df[df['label'] == label]['text'])
    wc = WordCloud(width=700, height=400, background_color='white',
                   colormap=color, max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Train & Compare Models

In [ ]:
best_model, results, X_test, y_test = train_and_evaluate(df)
results

In [ ]:
# Model comparison bar chart
fig, ax = plt.subplots(figsize=(12, 5))
results[['accuracy','precision','recall','f1']].plot(kind='bar', ax=ax,
    color=['#3498db','#e74c3c','#2ecc71','#f39c12'], edgecolor='white', linewidth=0.5)
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
ax.set_xticklabels(results.index, rotation=20)
plt.tight_layout()
plt.show()

## 4. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test,
    display_labels=['Ham', 'Spam'],
    cmap='Blues', ax=ax
)
ax.set_title('Best Model — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Live Predictions

In [ ]:
test_emails = [
    "Congratulations! You've won a FREE iPhone. Claim now!",
    "Hi, are you free for a quick catch-up call tomorrow?",
    "URGENT: Your bank account has been compromised. Verify immediately.",
    "Please find attached the meeting notes from today.",
    "Make $5000 a week working from home! Limited time offer!",
]

results_pred = predict(best_model, test_emails)
for r in results_pred:
    print(f"{r['prediction']:15} | {r['text']}")